In [ ]:
%%capture
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from datasets import load_dataset
import urllib.request

# Download your data
urllib.request.urlretrieve("https://raw.githubusercontent.com/Youmei295/DeAIze-model-source-code/main/dataset.json", "dataset.json")
dataset = load_dataset("json", data_files="dataset.json", split="train")

def formatting_func(examples):
    texts = []
    for inp, tar in zip(examples["input"], examples["target"]):
        messages = [
            {"role": "system", "content": "You are a professional Vietnamese editor. Rewrite the text to be natural, human-sounding, and concise. Remove robotic AI-isms and repetitive transitions."},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": tar},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Define your Training Arguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 5e-5,
    fp16 = True,
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

# 2. Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

# 3. Train the model
trainer.train()

In [ ]:
# Save and push the LoRA adapters to the Hugging Face Hub
model.push_to_hub("Youmei295/deAIze", token=hf_token)
tokenizer.push_to_hub("Youmei295/deAIze", token=hf_token)